# LizyML Tutorial: Binary Classification with LightGBM

This notebook demonstrates the full LizyML workflow on a **binary classification** task:

1. Data preparation (German Credit dataset from UCI / OpenML)
2. Config setup (StratifiedKFold default, early stopping)
3. Model fit (5-fold stratified CV with early stopping)
4. Evaluate — metrics table + learning curve
5. ROC Curve (IS vs OOS)
6. Confusion Matrix (IS vs OOS)
7. Calibration — Platt scaling, reliability diagram, probability histogram
8. Feature importance — Split, Gain, SHAP

**Dataset**: [German Credit (credit-g)](https://www.openml.org/d/31) — 1,000 rows, 20 features (numeric + categorical)  
**Target**: `credit_risk` (0 = good, 1 = bad)

## 1. Setup

In [ ]:
from __future__ import annotations

import pandas as pd
from sklearn.datasets import fetch_openml

from lizyml import Model

## 2. Data Preparation

Download the [German Credit dataset](https://www.openml.org/d/31) from OpenML.
This classic UCI dataset predicts credit risk (good/bad) from 20 features including
credit history, loan purpose, employment status, and more.

In [ ]:
credit = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")

df = credit.data.copy()
# Convert categorical columns to string for auto_categorical
for col in df.select_dtypes(include=["category"]).columns:
    df[col] = df[col].astype(str)
df["credit_risk"] = (credit.target == "bad").astype(int)

print(f"Shape: {df.shape}")
print(f"Bad credit rate: {df['credit_risk'].mean():.1%}")
df.head()

In [ ]:
df.describe()

## 3. Config

All 14 LightGBM parameters are explicitly specified as a tutorial reference:

- **model.params** (9): native LightGBM parameters — `objective`, `n_estimators`, `learning_rate`, `max_depth`, `feature_fraction`, `bagging_fraction`, `bagging_freq`, `lambda_l2`, `max_bin`
- **smart params** (3): data-size-relative — `num_leaves_ratio`, `min_data_in_leaf_ratio`, `min_data_in_bin_ratio` (resolved at fit time via `auto_num_leaves`)
- **training** (2): early stopping — `rounds`, `validation_ratio`

For binary classification, `split` is omitted — LizyML defaults to `stratified_kfold` (H-0013).
`balanced=True` computes `scale_pos_weight` from class distribution.

In [ ]:
config = {
    "config_version": 1,
    "task": "binary",
    "data": {
        "target": "credit_risk",
    },
    "model": {
        "name": "lgbm",
        # --- Native LightGBM parameters ---
        "params": {
            "objective": "binary",
            "n_estimators": 1500,
            "learning_rate": 0.001,
            "max_depth": 5,
            "feature_fraction": 0.7,
            "bagging_fraction": 0.7,
            "bagging_freq": 10,
            "lambda_l2": 0.000001,
            "max_bin": 511,
        },
        # --- Smart parameters (resolved at fit time) ---
        "auto_num_leaves": True,
        "num_leaves_ratio": 1.0,
        "min_data_in_leaf_ratio": 0.01,
        "min_data_in_bin_ratio": 0.001,
        # balanced=True computes scale_pos_weight from class distribution
        "balanced": True,
    },
    "features": {
        "auto_categorical": True,  # auto-detects categorical columns
    },
    "training": {
        "seed": 42,
        "early_stopping": {
            "enabled": True,
            "rounds": 150,
            "validation_ratio": 0.1,
        },
    },
    "evaluation": {
        "metrics": ["logloss", "auc", "auc_pr", "f1", "accuracy", "brier", "ece"],
    },
    "calibration": {
        "method": "platt",
        "n_splits": 5,
    },
}

## 4. Model Fit

In [ ]:
model = Model(config)
model.fit(data=df)
print("Fit complete.")

### 4.1 LightGBM Parameters

Config smart params and resolved booster params.

In [ ]:
model.params_table()

## 5. Evaluate

### 5.1 Metrics Table

In [ ]:
model.evaluate_table().round(4)

### 5.2 Learning Curve

In [ ]:
model.plot_learning_curve().show()

## 6. ROC Curve (IS vs OOS)

In [ ]:
model.roc_curve_plot().show()

## 7. Confusion Matrix (IS vs OOS)

In [ ]:
cm = model.confusion_matrix(threshold=0.5)
print("=== OOS Confusion Matrix ===")
display(cm["oos"])
print("\n=== IS Confusion Matrix ===")
display(cm["is"])

## 8. Calibration

Platt scaling was enabled in the config. Compare raw vs calibrated OOF probabilities.

### 8.1 Calibration Curve (Reliability Diagram)

In [ ]:
model.calibration_plot().show()

### 8.2 Probability Histogram (Raw vs Calibrated)

In [ ]:
model.probability_histogram_plot().show()

## 9. Feature Importance: Split

In [ ]:
model.importance_plot(kind="split").show()

In [ ]:
pd.Series(model.importance(kind="split"), name="importance_split").sort_values(
    ascending=False
).to_frame()

## 10. Feature Importance: Gain

In [ ]:
model.importance_plot(kind="gain").show()

## 11. Feature Importance: SHAP

In [ ]:
model.importance_plot(kind="shap").show()

In [ ]:
pd.Series(model.importance(kind="shap"), name="mean_abs_shap").sort_values(
    ascending=False
).to_frame()